# ✍️ Homework — `annotate_gene.R`

Today you will write **`annotate_gene.R`**, which takes `counts.tsv` in and writes **`counts_annotated.tsv`** out — with a `Symbol` column glued on and the 12 sample columns renamed to something you can read.

## Fetch the raw GEO counts

In [ ]:
dir.create("counts", showWarnings = FALSE)      # create it once

gz  <- "counts/counts.tsv.gz"
tsv <- "counts/counts.tsv"

url <- paste0("https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts",
              "&acc=GSE129221&format=file",
              "&file=GSE129221_raw_counts_GRCh38.p13_NCBI.tsv.gz")

system2("curl",   c("-sS", "-o", shQuote(gz), shQuote(url)))
system2("gunzip", c("-f", shQuote(gz)))

## 📖 What You Already Know

Everything here is Lecture 6 material:

| You met it | You need it for |
|:--|:--|
| `setwd()`, `getwd()`, `list.files()` | finding the files |
| `read.delim()`, `row.names = 1` | reading the counts |
| `head()`, `dim()`, `str()`, `colnames()` | inspecting the reference |
| `rep()`, `paste()` | naming the 12 samples |
| `match()` | the actual join |
| `is.na()`, `sum()` | checking for unmatched genes |
| `data.frame()`, `$` | building the output table |
| `write.table()` | saving it |
| `stopifnot()` | refusing to write a broken file |

## Build `sample_metadata.csv`

Before we can name a single column, we need to know what the 12 samples **are**. The GEO accessions say nothing — `GSM3702795` could be anything, and `PC9` versus `PC9GR` (gefitinib-**r**esistant) is completely invisible in a number.

In [ ]:
# --- read the raw counts -----------------------------------------------
counts <- read.delim("counts/counts.tsv", row.names = 1)

dim(counts)                  # 39376 12
colnames(counts)              # GSM3702795 ... GSM3702806
head(rownames(counts))

# --- Step 1: the sample IDs --------------------------------------------
sample_id <- _______________________

# --- Step 2: the cell line ---------------------------------------------
cell_line <- _______________________

# --- Step 3: the treatment ---------------------------------------------
treatment <- _______________________

# --- Step 4: the replicate ---------------------------------------------
replicate <- _______________________

# --- four vectors -> one data frame ------------------------------------
sample_info <- data.frame(
  sample_id = sample_id,
  cell_line = cell_line,
  treatment = treatment,
  replicate = replicate
)

sample_info

# --- save it: row.names = FALSE drops the 1..12 row numbers ------------
write.csv(sample_info, "counts/sample_metadata.csv", row.names = FALSE)

file.exists("counts/sample_metadata.csv")

<details>
<summary><b>✅ Answer</b></summary>

```r
counts <- read.delim("counts/counts.tsv", row.names = 1)

# Step 1: the sample IDs
sample_id <- colnames(counts)

# Step 2: the cell line
cell_line <- rep(c("PC9", "PC9GR"), each = 6)

# Step 3: the treatment
treatment <- rep(c("Control", "Apatinib"), times = 2, each = 3)

# Step 4: the replicate
replicate <- rep(1:3, times = 4)

sample_info <- data.frame(
  sample_id = sample_id,
  cell_line = cell_line,
  treatment = treatment,
  replicate = replicate
)

write.csv(sample_info, "counts/sample_metadata.csv", row.names = FALSE)
```

</details>

## Merge colnames

One line joins the table to the matrix: `paste()` glues the three length-12 vectors into names like `PC9_Control_1`, `PC9_Control_2`, ...


In [ ]:
# --- Name the 12 sample columns ----------------------------------------
colnames(counts) <- ______________________________

colnames(counts)
head(counts)

# --- Save the renamed matrix --------------------------------------------
write.table(counts, ______________________)

file.exists("counts/counts_sample_name_merge.tsv")

<details>
<summary><b>✅ Answer</b></summary>

```r
colnames(counts) <- paste(sample_info$cell_line,
                          sample_info$treatment,
                          sample_info$replicate, sep = "_")

write.table(counts, "counts/counts_sample_name_merge.tsv",
            sep = "\t", row.names = TRUE)
```

</details>

---

## Annotate the Gene IDs

That gives us `counts_sample_name_merge.tsv` — genes by named sample — but the first column is still a wall of Entrez numbers. To read it we need NCBI's annotation file, `Human.GRCh38.p13.annot.tsv`, which is what maps `100287102` onto `DDX11L1`.

`curl` misbehaved on this one, so take it by hand:

1. Click [Human.GRCh38.p13.annot.tsv.gz](https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz) — the browser saves the compressed file.
2. Decompress it: macOS/Linux right-click → *Decompress*; Windows use 7-Zip → *Extract here*. You should now have `Human.GRCh38.p13.annot.tsv`.
3. Open the **Files** panel (folder icon, left sidebar) and drag that `.tsv` into `counts/`.
4. Confirm it arrived with `list.files("counts")`.


## Look before you leap

Two files, two very different shapes. Open both and look at the header row before writing any code.


In [ ]:
# --- look at both files before trusting either -------------------------
list.files("counts")

head("counts/counts_sample_name_merge.tsv", 2)
head("counts/Human.GRCh38.p13.annot.tsv", 2)

[1] "counts_sample_name_merge.tsv" "counts.tsv"                  
[3] "Human.GRCh38.p13.annot.tsv"   "sample_metadata.csv"

[1] "counts/counts_sample_name_merge.tsv"

[1] "counts/Human.GRCh38.p13.annot.tsv"

Count the cells in each header row.

The counts file names its 12 samples and **stops** — the Entrez ID column has no header cell of its own, so the header row carries 12 cells while every data row carries 13. That gap is exactly why Step 2 reads it with `row.names = 1`.

The annotation file is the opposite: all 18 columns are named, `GeneID`, `Symbol`, `Description`, … A proper lookup table.


## 1. Point at the files

Name the folder once, then build every path from it. That way only the first line ever changes if the data moves. The original script used `here()` for the same job — that needs a real project root, which `/content` is not.


In [ ]:
counts_dir <- "counts"

counts_file <- file.path(counts_dir, ______________________)
annot_file  <- file.path(counts_dir, ______________________)
out_file    <- file.path(counts_dir, ______________________)

<details>
<summary><b>✅ Answer</b></summary>

```r
counts_dir <- "counts"

counts_file <- file.path(counts_dir, "counts_sample_name_merge.tsv")
annot_file  <- file.path(counts_dir, "Human.GRCh38.p13.annot.tsv")
out_file    <- file.path(counts_dir, "counts_annotated.tsv")
```

</details>


## 2. Read both files

Both are tab-separated and both are read the same way, but the counts file needs one extra thing: its first column has to become the row labels, because that column is the Entrez ID and nothing else.


In [ ]:
# --- 2. read both files ------------------------------------------------
# row.names = ... drops the first column into the row labels.
# Without it you get an extra column and rownames(counts) becomes "1","2","3"...

counts <- read.delim(______________________)
annot  <- read.delim(______________________)

dim(counts)
colnames(counts)

<details>
<summary><b>✅ Answer</b></summary>

```r
counts <- read.delim(counts_file, row.names = 1)
annot  <- read.delim(annot_file)

dim(counts)       # 39376 12
colnames(counts)   # PC9_Control_1 ... PC9GR_Apatinib_3
```

</details>


## 3. Find each gene in the reference

`match()` answers with *positions*, not values, and never reorders anything. `merge()` would sort the genes alphabetically and quietly break the correspondence with every other object keyed by those same rows.


In [ ]:
# --- 3. find each gene in the reference --------------------------------
# match() takes what to look for, and where to look.
# It answers with POSITIONS, not values.

idx <- ______________________
# these two lines should print the SAME genes - that is the whole point
rownames(counts)[1:3]
annot$GeneID[idx][1:3]

<details>
<summary><b>✅ Answer</b></summary>

```r
idx <- match(rownames(counts), annot$GeneID)

# rownames(counts)[1:3]  -> "100287102" "653635" "102466751"
# annot$GeneID[idx][1:3] -> "100287102" "653635" "102466751"
```

</details>


## 4. Check nothing was lost *(optional)*

`match()` gives back `NA` for anything it cannot find, so the number of `NA`s is exactly the number of genes that would have been written blank. Refuse to continue rather than ship that.


In [ ]:
# --- 4. check nothing was lost (optional) -------------------------------
# is.na() flags the misses, ! flips them, sum() counts.

cat("genes in counts      :", nrow(counts), "\n")
cat("matched in reference :", ______________________, "\n")
cat("unmatched            :", ______________________, "\n")

stopifnot(______________________)

<details>
<summary><b>✅ Answer</b></summary>

```r
cat("genes in counts      :", nrow(counts), "\n")
cat("matched in reference :", sum(!is.na(idx)), "\n")
cat("unmatched            :", sum(is.na(idx)), "\n")

stopifnot(all(!is.na(idx)))
```

</details>


## 5. Attach the annotation

Three pieces go into one frame: the IDs recovered from the row labels, the symbols fetched through the index, and the counts themselves untouched. Two arguments earn their place — one clears the old row labels now that `GeneID` is a real column, the other stops R rewriting `PC9_Control_1` into `PC9.Control.1`.


In [ ]:
# --- 5. attach the annotation ------------------------------------------
# Three pieces: GeneID from the row labels, Symbol via idx, then counts.

annotated <- data.frame(
  GeneID      = ______________________,
  Symbol      = ______________________,
  counts,
  row.names = ______________________,
  check.names = ______________________
)

dim(annotated)
head(annotated[, 1:5])

<details>
<summary><b>✅ Answer</b></summary>

```r
annotated <- data.frame(
  GeneID      = rownames(counts),
  Symbol      = annot$Symbol[idx],
  counts,
  row.names = NULL,
  check.names = FALSE
)

dim(annotated)   # 39376 14
```

</details>


## 6. Write it out

Tab-separated, no quotes, and no repeated row labels — `GeneID` is already the first column, so writing the labels out too would duplicate them.


In [ ]:
# --- 6. write it out ----------------------------------------------------
# tab separated, and do NOT repeat the row labels.

write.table(annotated, out_file, ______________________)
file.exists(out_file)

# the real proof: read it back off disk
check <- read.delim(out_file)
dim(check)
head(check, 2)

<details>
<summary><b>✅ Answer</b></summary>

```r
write.table(annotated, out_file, sep = "\t", row.names = FALSE, quote = FALSE)

# dim(check) -> 39376 14
```

</details>


---

## The full picture

Every step above, assembled. Save it as `annotate_gene.R` and run it on its own — it rebuilds `counts_annotated.tsv` from scratch every time, and it is the file you hand in.


In [ ]:
# annotate_gene.R
#
# Input : counts_sample_name_merge.tsv   (gene-by-sample, columns already renamed)
# Ref   : Human.GRCh38.p13.annot.tsv      (GeneID -> Symbol, Description, ...)
# Output: counts_annotated.tsv
#
# The counts matrix is keyed by Entrez GeneID in its row names. To attach
# symbols we look up each row name in the reference and keep the original
# row order - match() does that; merge() would re-sort and lose it.

counts_dir <- "counts"
counts_file <- file.path(counts_dir, "counts_sample_name_merge.tsv")
annot_file  <- file.path(counts_dir, "Human.GRCh38.p13.annot.tsv")
out_file    <- file.path(counts_dir, "counts_annotated.tsv")

counts <- read.delim(counts_file, row.names = 1)
annot  <- read.delim(annot_file)

# Where does each counted gene sit in the reference table?
idx <- match(rownames(counts), annot$GeneID)

cat("genes in counts      :", nrow(counts), "\n")
cat("matched in reference :", sum(!is.na(idx)), "\n")
cat("unmatched            :", sum(is.na(idx)), "\n")

stopifnot(all(!is.na(idx)))   # stop early rather than write a half-annotated file

# Attach the annotation, keeping GeneID as the first column.
annotated <- data.frame(
  GeneID      = rownames(counts),
  Symbol      = annot$Symbol[idx],
  # Description = annot$Description[idx],
  # GeneType    = annot$GeneType[idx],
  # EnsemblGeneID = annot$EnsemblGeneID[idx],
  counts,
  row.names = NULL,
  check.names = FALSE
)

write.table(annotated, out_file, sep = "\t", row.names = FALSE, quote = FALSE)
cat("written:", out_file, "\n")

genes in counts      : 39376 
matched in reference : 39376 
unmatched            : 0 
written: counts/counts_annotated.tsv 


# ✍️ Practice Exercises

Three exercises to close the lecture. Type your answer into the cell, then open the **✅ Answer** when you are done.

---

## ✍️ Exercise 1 — Lists & Double Brackets

A QC report came back from the pipeline as a list:

1. Extract `umi_counts` **as a vector** (not wrapped in a list).
2. Compute the mean UMI count.
3. How many spots fall below `100` reads?

In [ ]:
qc_report <- list(
  sample_name = "Visium_HD_01",
  umi_counts  = c(200, 450, 12, 90, 800),
  pass_status = TRUE
)

# 1. counts   <- qc_report[["umi_counts"]]
# 2. mean(counts)
# 3. sum(counts < 100)

<details>
<summary><b>✅ Answer</b></summary>

```r
counts <- qc_report[["umi_counts"]]   # [[ ]] unwraps the vector

mean(counts)          # mean library depth per spot

sum(counts < 100)     # how many low-quality spots?
```

</details>

---

## ✍️ Exercise 2 — apply() & Missing Data

A gene expression matrix arrived with holes in it:

1. Replace every `NA` with `0`.
2. Compute the **median of each column** with `apply()`.
3. Print the column with the highest median.

In [ ]:
test_data <- matrix(c(5, 10, NA, 2, 8, 12, 1, NA, 3), nrow = 3, ncol = 3)
test_data

# 1. test_data[is.na(test_data)] <- 0
# 2. col_medians <- apply(test_data, 2, median)
# 3. which.max(col_medians)

<details>
<summary><b>✅ Answer</b></summary>

```r
# 1. fill the holes
test_data[is.na(test_data)] <- 0

# 2. MARGIN 2 -> one median per column
col_medians <- apply(test_data, 2, median)

# 3. which column wins?
which.max(col_medians)
```

`which.max()` returns a **position**, not a value - the same idea as `grep()`.

</details>

---

## ✍️ Exercise 3 — Clean the Gene List

1. Trim the whitespace and force everything to upper case.
2. Drop the `-mouse` suffix.
3. Report how many genes remain and print them.

In [ ]:
genes <- c("gapdh", "ACTB-mouse", "Cd8a-MOUSE", "il6 ", "PECAM1-mouse")
genes

# 1. genes <- toupper(trimws(genes))
# 2. genes <- gsub("-MOUSE", "", genes)
# 3. length(genes) ; genes

<details>
<summary><b>✅ Answer</b></summary>

```r
# 1. whitespace + case
genes <- toupper(trimws(genes))

# 2. strip the suffix
genes <- gsub("-MOUSE", "", genes)

# 3. count and print
length(genes)
genes
```

Clean the **case first**, then the pattern - otherwise you need one regex per capitalisation.

</details>